In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 10)

In [ ]:
try:
    driver.get("http://localhost:5173/")
    driver.execute_script("window.localStorage.clear(); window.sessionStorage.clear();")
    driver.get("http://localhost:5173/login")

    # Login selectors verified in login.jsx / LoginInput.jsx / LoginButton.jsx
    wait.until(EC.presence_of_element_located((By.ID, "username")))
    driver.find_element(By.ID, "username").send_keys("shawon@gmail.com")
    driver.find_element(By.ID, "password").send_keys("12345678")
    driver.find_element(By.ID, "sign-in-btn").click()
    time.sleep(3)

    # Real pharmacist home route (verified: roleHomePath in services/auth.js)
    wait.until(EC.visibility_of_element_located((By.XPATH, "//nav[@aria-label='Staff']")))
    assert "/pharmacist/dashboard" in driver.current_url, f"Unexpected landing URL: {driver.current_url}"

    # Navigate directly to the real route with driver.get
    driver.get("http://localhost:5173/pharmacist/dashboard")
    wait.until(EC.visibility_of_element_located((By.XPATH, "//nav[@aria-label='Staff']")))
    time.sleep(2)
    body = driver.find_element(By.TAG_NAME, "body").text
    assert len(body.strip()) > 200, "Page is blank after direct URL access."
    assert not driver.find_elements(By.ID, "username"), "Logged out to the login form."
    token = driver.execute_script("return window.localStorage.getItem('pharvo_access_token');")
    assert token, "Access token missing after direct URL access."
    print("Direct URL loaded:", driver.current_url)
    print("PASS: Direct URL access works")
except Exception as e:
    print("FAIL:", e)
    driver.save_screenshot("46_direct_url_FAIL.png")
finally:
    driver.quit()